In [8]:
import requests
import time
from datetime import datetime
from collections import deque
import plotly.graph_objects as go
from IPython.display import display, clear_output
import gspread
from google.oauth2.service_account import Credentials
from google.colab import files  # Colab-specific import for file upload

# Step 1: Upload your Google Service Account JSON file
print("Please upload your Google Service Account JSON file.")
uploaded = files.upload()  # This will prompt you to upload the file
credentials_file = list(uploaded.keys())[0]  # Use the uploaded file's name

# Configuration
data_url = 'https://6fc0-2405-201-6815-261-31fc-f5ec-fd9a-d83b.ngrok-free.app/data'  # Replace with your Ngrok URL
num_channels = 1  # Number of EEG channels
buffer_size = 512  # Number of points to show on the graph
signal_buffers = [deque([0] * buffer_size, maxlen=buffer_size) for _ in range(num_channels)]

# Initialize Plotly figure
fig = go.Figure()

# Add traces for all channels to a single graph
for i in range(num_channels):
    fig.add_trace(go.Scatter(y=[], mode='lines', name=f'Channel {i}'))

fig.update_layout(
    title="Real-time EEG Signal Visualization",
    xaxis_title="Time",
    yaxis_title="Signal Value",
    xaxis=dict(range=[0, buffer_size]),  # Fixed x-axis range
    yaxis=dict(),         # Adjust based on sensor value range
    showlegend=True,
)

# Step 2: Set up Google Sheets
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
credentials = Credentials.from_service_account_file(credentials_file, scopes=scope)
client = gspread.authorize(credentials)

# Open the existing Google Sheet by its key
sheet_key = "175qYNdMyPRX4qW-1kQLZKT9Pz6JQwEJ7Rqi0H5z5vAs"
sheet = client.open_by_key(sheet_key).sheet1

# Write column headers to the Google Sheet
headers = ["Time"] + [f"A{i}" for i in range(num_channels)]
sheet.clear()  # Clear any existing data in the sheet
sheet.append_row(headers)  # Add headers

# Buffer for batch writing to Google Sheets
batch_buffer = []
BATCH_SIZE = 10  # Number of data points to collect before writing to the sheet

# Step 3: Real-time data fetching and plotting
try:
    while True:
        try:
            # Fetch data from the server
            response = requests.get(data_url, timeout=5)
            if response.status_code == 200:
                values = response.json()

                # Ensure valid data format
                if len(values) == num_channels:
                    # Add the current timestamp to the data
                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    row_with_time = [timestamp] + values

                    # Update buffers with new data
                    for i in range(num_channels):
                        signal_buffers[i].append(values[i])

                    # Update Plotly data
                    for i in range(num_channels):
                        fig.data[i].y = list(signal_buffers[i])

                    # Add data to batch buffer for Google Sheets
                    batch_buffer.append(row_with_time)

                    # If batch buffer reaches the defined size, write to Google Sheets
                    if len(batch_buffer) >= BATCH_SIZE:
                        print("Writing batch data to Google Sheets...")
                        try:
                            # Write the batch data to the sheet
                            sheet.append_rows(batch_buffer)  # Write all rows in the batch
                            batch_buffer = []  # Clear the buffer after writing
                        except gspread.exceptions.APIError as e:
                            print(f"Error writing to Google Sheets: {e}")
                            time.sleep(10)  # Wait for a while before retrying

                    # Render the updated figure in Colab without flicker
                    clear_output(wait=True)
                    display(fig)

                    # Reduce update frequency to prevent flickering
                    time.sleep(0.2)  # Adjust this value to control the smoothness

                else:
                    print("Invalid data format received:", values)
            else:
                print(f"Failed to fetch data. Status code: {response.status_code}")
        except requests.exceptions.RequestException as e:
            print("Error fetching data:", e)
            time.sleep(1)  # Retry after a delay
except KeyboardInterrupt:
    print("Real-time visualization stopped.")


Real-time visualization stopped.
